# Delta Lake Exploration

Interactive notebook for querying Bronze, Silver, and Gold Delta tables.

## Bronze Layer Validation
Verify ingestion, schema, metadata, data quality, and checkpoint state.

In [ ]:
from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

In [ ]:
bronze_path = config["paths"]["delta_bronze"]
bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Bronze row count: {bronze_df.count()}")
print(f"Columns: {bronze_df.columns}")
bronze_df.printSchema()
bronze_df.show(10, truncate=False)

### Bronze Data Quality Checks

In [ ]:
from pyspark.sql.functions import (
    col, count, min as spark_min, max as spark_max,
    avg, stddev, countDistinct,
)

print("=== Null / Corrupt Check ===")
bronze_df.select(
    count("*").alias("total_rows"),
    count(col("device_id")).alias("non_null_device_id"),
    count(col("timestamp")).alias("non_null_timestamp"),
    count(col("_corrupt_record")).alias("corrupt_records"),
    countDistinct("device_id").alias("distinct_devices"),
    countDistinct("_source_file").alias("distinct_files"),
).show(truncate=False)

print("=== Numeric Field Stats ===")
bronze_df.select(
    spark_min("temperature").alias("temp_min"),
    spark_max("temperature").alias("temp_max"),
    avg("temperature").alias("temp_avg"),
    spark_min("humidity").alias("hum_min"),
    spark_max("humidity").alias("hum_max"),
    avg("humidity").alias("hum_avg"),
    spark_min("pressure").alias("pres_min"),
    spark_max("pressure").alias("pres_max"),
).show(truncate=False)

print("=== Ingestion Metadata ===")
bronze_df.select(
    spark_min("_ingested_at").alias("earliest_ingestion"),
    spark_max("_ingested_at").alias("latest_ingestion"),
    spark_min("timestamp").alias("earliest_event"),
    spark_max("timestamp").alias("latest_event"),
).show(truncate=False)

### Delta Table History & Versions

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, bronze_path)
print("=== Delta Version History ===")
dt.history().show(truncate=False)

print("=== Current Table Detail ===")
detail = dt.detail().select(
    "format", "location", "numFiles",
    "sizeInBytes", "partitionColumns",
).first()
print(f"  Format:      {detail['format']}")
print(f"  Location:    {detail['location']}")
print(f"  Num files:   {detail['numFiles']}")
print(f"  Size (bytes):{detail['sizeInBytes']}")
print(f"  Partitions:  {detail['partitionColumns']}")

### Bronze - Per-Device Breakdown

In [ ]:
device_stats = (
    bronze_df
    .groupBy("device_id", "location")
    .agg(
        count("*").alias("events"),
        avg("temperature").alias("avg_temp"),
        avg("humidity").alias("avg_humidity"),
        avg("battery_level").alias("avg_battery"),
    )
    .orderBy(col("events").desc())
)
print(f"=== Per-Device Stats ({device_stats.count()} devices) ===")
device_stats.show(20, truncate=False)

## Silver Layer

In [ ]:
silver_path = config["paths"]["delta_silver"]
silver_df = spark.read.format("delta").load(silver_path)

print(f"Silver row count: {silver_df.count()}")
silver_df.printSchema()

silver_df.select(
    count("*").alias("total"),
    countDistinct("device_id").alias("devices"),
    avg("_quality_score").alias("avg_quality"),
    count(col("_is_anomaly").cast("int")).alias("anomalies_flagged"),
).show(truncate=False)

silver_df.select(
    "device_id", "temperature", "humidity", "_is_anomaly",
    "_temperature_zscore", "_quality_score",
).show(10, truncate=False)

In [ ]:
from delta.tables import DeltaTable as SilverDT

sdt = SilverDT.forPath(spark, silver_path)
print("=== Silver Delta History ===")
sdt.history().select("version", "timestamp", "operation").show(truncate=False)

## Gold Layer

In [ ]:
from pathlib import Path

gold_base = config["paths"]["delta_gold"]
gold_tables = ["device_summary", "anomaly_summary", "device_health", "ml_features"]

for table in gold_tables:
    path = str(Path(gold_base) / table)
    try:
        gdf = spark.read.format("delta").load(path)
        print(f"=== {table} ({gdf.count()} rows, {len(gdf.columns)} cols) ===")
        gdf.show(5, truncate=False)
    except Exception as e:
        print(f"=== {table}: not available ({e}) ===")